In [15]:
import pandas as pd
import numpy as np

In [16]:
merged_data=pd.read_csv('Merged_Data.csv')
customers=pd.read_csv('Customers.csv')

In [19]:
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import StandardScaler

# Feature Engineering
# Aggregate transaction data by CustomerID
customer_features = merged_data.groupby('CustomerID').agg({
    'TotalValue': 'sum',          # Total transaction value
    'Quantity': 'sum',            # Total quantity purchased
    'TransactionPrice': 'mean'    # Average price during transactions
}).reset_index()

# Add profile features from Customers.csv
customer_features = customer_features.merge(customers, on='CustomerID', how='left')

# Normalize features
scaler = StandardScaler()
numeric_columns = customer_features.select_dtypes(include=['number']).columns
scaled_features = scaler.fit_transform(customer_features[numeric_columns])


# Compute similarity
similarity_matrix = cosine_similarity(scaled_features)

# Generate Lookalikes for C0001-C0020
lookalikes = {}
for i, customer_id in enumerate(customer_features['CustomerID'][:20]):
    similar_indices = np.argsort(similarity_matrix[i])[-4:][::-1]  # Exclude the customer themselves
    similar_customers = [
        (customer_features['CustomerID'][idx], similarity_matrix[i][idx])
        for idx in similar_indices
        if idx != i
    ]
    lookalikes[customer_id] = similar_customers[:3]

# Save Lookalike CSV
lookalike_data = []
for customer_id, similar_data in lookalikes.items():
    row = [customer_id] + [item for pair in similar_data for item in pair]
    lookalike_data.append(row)

lookalike_df = pd.DataFrame(lookalike_data, columns=[
    'CustomerID', 'SimilarCustomer1', 'Score1', 'SimilarCustomer2', 'Score2', 'SimilarCustomer3', 'Score3'
])
lookalike_df.to_csv('Rishabh_Kumar_Lookalike.csv', index=False)
